# Практическая работа 6. Методы первого и второго порядка в оптимизации

## Цель работы
Реализовать и сравнить классические методы первого и второго порядка для задач гладкой и композитной оптимизации.

## Содержание
1. Методы первого порядка (Gradient Descent, SGD, Momentum, Accelerated GD)
2. Адаптивные методы (Adam, RMSProp)
3. Композитная оптимизация (ISTA, FISTA)
4. Методы второго порядка (Ньютон, BFGS)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits, make_circles
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd

## Задание 1: GD на квадратичной функции

**Теория**: Градиентный спуск использует итерацию:
$$x_{k+1} = x_k - \eta \nabla f(x_k)$$

Для сильно выпуклой функции скорость сходимости зависит от числа обусловленности $\kappa = L/\mu$.

In [ ]:
# Задание 1: GD на квадратичной функции
# Задача: f(x) = 1/2 x^T Q x + b^T x, где Q - плохо обусловлена

def create_quadratic_problem(d=20, kappa=100):
    """Создать квадратичную задачу с числом обусловленности kappa"""
    eigvals = np.logspace(0, np.log10(kappa), d)
    Q = np.diag(eigvals)
    b = np.random.randn(d)
    
    def f(x):
        return 0.5 * x @ Q @ x + b @ x
    
    def grad_f(x):
        return Q @ x + b
    
    def hess_f(x):
        return Q
    
    return f, grad_f, hess_f, Q, b

# Создание задачи
d = 10
kappa = 100
f_quad, grad_quad, hess_quad, Q, b = create_quadratic_problem(d, kappa)
L = np.max(np.diag(Q))  # Константа гладкости
x_opt = -np.linalg.inv(Q) @ b  # Оптимальное решение

# Градиентный спуск
def gradient_descent(grad_f, x0, learning_rates, max_iters=500):
    """GD с несколькими learning rates"""
    results = {}
    for eta in learning_rates:
        x = x0.copy()
        losses = [f_quad(x)]
        for k in range(max_iters):
            x = x - eta * grad_f(x)
            losses.append(f_quad(x))
        results[eta] = losses
    return results

x0 = np.random.randn(d)
learning_rates = [0.001, 1/L, 0.01, 0.1]
gd_results = gradient_descent(grad_quad, x0, learning_rates, max_iters=200)

# Визуализация
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# График функции
ax = axes[0]
for eta, losses in gd_results.items():
    ax.semilogy(losses, label=f'η = {eta:.4f}')
ax.set_xlabel('Итерация')
ax.set_ylabel('f(x_k)')
ax.set_title('GD на квадратичной функции')
ax.legend()
ax.grid(True, alpha=0.3)

# Расстояние до оптимума
ax = axes[1]
for eta, losses in gd_results.items():
    x = x0.copy()
    distances = [np.linalg.norm(x - x_opt)]
    for k in range(len(losses) - 1):
        x = x - eta * grad_quad(x)
        distances.append(np.linalg.norm(x - x_opt))
    ax.semilogy(distances, label=f'η = {eta:.4f}')
ax.set_xlabel('Итерация')
ax.set_ylabel('||x_k - x*||')
ax.set_title('Расстояние до оптимума')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("=" * 70)
print("ЗАДАНИЕ 1: GD на квадратичной функции")
print("=" * 70)
print(f"Число обусловленности κ = {kappa}")
print(f"Оптимальный шаг: η = 1/L = {1/L:.6f}")
print(f"Число обусловленности матрицы Q: {np.linalg.cond(Q):.2f}")
print("\nРезультаты после 200 итераций:")
for eta in learning_rates:
    final_loss = gd_results[eta][-1]
    status = "✓" if final_loss < 0.1 else "✗" if not np.isfinite(final_loss) else "~"
    print(f"  η = {eta:.4f}: f(x) = {final_loss:.6e} {status}")